# Obtain Data

This Notebook is pretended to be used as an extractor and pre-processor of the data required for the Film Communities Network Science Project.

The main process is as follows:
- Extract Reddit posts ids from Movies.html (An HTML version of the docx file cotaining all pertinent posts)
- Extract comments from each post and save them as CSV

## Imports

In [1]:
from pathlib import Path

import pandas as pd
import praw

from utils.id_extractor import get_ids_list
from utils.reddit_utils import fetch_comments_from_posts

## Environment setting

In [2]:
reddit = praw.Reddit(
        client_id = '',
        client_secret = '',
        user_agent = '',
    )

comments_path = Path("./data/comments")

## Get Post ids

In [ ]:
ids = get_ids_list("./data/Movies.html")
id_movie_df = pd.DataFrame(ids, columns=["post_id", "movie"])
id_movie_df

,post_id,movie
0,1gwxxy1,Wicked
1,1gv6e9j,Wicked
2,1grc9di,Wicked
3,ucg7gt,Wicked
4,1gx1k96,Wicked
...,...,...
455,zr8e38,The Whale
456,119maug,The Whale
457,10ut5s4,The Whale
458,1061zz6,The Whale


#### Only use this chunk if you do not want to redownload posts that have already been processed

In [9]:
processed_posts = [p.name.split(".")[0] for p in comments_path.iterdir() if p.is_file()]

# Update ids
ids = [x for x,_ in ids if x not in processed_posts]
ids_df = pd.DataFrame(ids, columns=["post_id"])

In [10]:
print(len(ids_df))

0


## Extract Data from Reddit

In [ ]:
for i in range(len(ids_df)):
    comments_df = fetch_comments_from_posts(reddit, ids_df.loc[[i]])

    comments_df.to_csv(f"{comments_path}/{ids_df["post_id"][i]}.csv")
    print(f"{ids_df["post_id"][i]}.csv was saved successfully.")

## Join all comments in one file

In [13]:
processed_files = [p for p in comments_path.iterdir() if p.is_file()]

full_comments = pd.DataFrame()

for i,file in enumerate(processed_files):
    file_df = pd.read_csv(file, index_col=0).reset_index(drop=True)
    full_comments = pd.concat([full_comments, file_df], axis=0)

full_comments = full_comments.merge(id_movie_df, on="post_id", how="left")

full_comments.to_csv("./data/full_comments.csv")

In [14]:
full_comments

,comment_id,post_id,author_id,author_name,body,score,created_utc,created_datetime,parent_id,parent_type,is_submitter,stickied,depth,controversiality,gilded,movie
0,ne2vqnm,1ngbiv2,"('8f2m70tk',)",tearsandpain84,More repo man vibes than QT,15,1.757807e+09,2025-09-14 01:37:43,1ngbiv2,post,False,False,0,0,0,Freaky Tales
1,ne2tk5w,1ngbiv2,"('qfaranj',)",nj_crc,This movie f&cks. Incredibly rewatchable. \n\n...,20,1.757806e+09,2025-09-14 01:25:18,1ngbiv2,post,False,False,0,0,0,Freaky Tales
2,ne335n2,1ngbiv2,"('4dsjmqck',)",TransportationAway59,Rip angus,6,1.757809e+09,2025-09-14 02:22:18,1ngbiv2,post,False,False,0,0,0,Freaky Tales
3,ne3693x,1ngbiv2,"('jsn1z4uu8',)",BeautifulLeather6671,I think it’s definitely more entertaining if y...,4,1.757810e+09,2025-09-14 02:41:37,1ngbiv2,post,False,False,0,0,0,Freaky Tales
4,ne38rtt,1ngbiv2,"('5mphk69',)",SeaWolf24,I love this movie,2,1.757811e+09,2025-09-14 02:56:54,1ngbiv2,post,False,False,0,0,0,Freaky Tales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137748,lrw1apy,1g3hq3r,"('9pos9hs',)",Timqwe,"As far as the movie itself goes, it's fine. \...",1,1.728920e+09,2024-10-14 17:34:48,lrvzxr9,comment,False,False,3,0,0,Hit Man
137749,lrwq4xv,1g3hq3r,"('qtkci',)",KhrysesAD,I haven't but I'll take a look now! 😂,1,1.728928e+09,2024-10-14 19:45:27,lrwmlur,comment,True,False,3,0,0,Hit Man
137750,lrw00j9,1g3hq3r,"('em14qa7',)",shobidoo2,Sort of. It would also be the same as 80 peopl...,4,1.728920e+09,2024-10-14 17:27:57,lrvz6sq,comment,False,False,4,0,0,Hit Man
137751,lrx97tv,1g3hq3r,"('7fr4rnj1',)",scottishhistorian,Just don't pay for them. If you find some of t...,2,1.728934e+09,2024-10-14 21:25:08,lrwq4xv,comment,False,False,4,0,0,Hit Man
